# KrakenSDR: Cyclic Reading & FFT (All Channels)

This notebook reads complex IQ samples from **all available RX channels** via **SoapySDR**, computes **FFT** per channel, and displays live **power spectra**.

If `SoapySDR` or the device is unavailable, the notebook automatically switches to a **simulation mode** that generates multi-tone signals per channel so you can verify the pipeline.

## Requirements
- `SoapySDR` runtime + Python bindings (`python3-soapysdr`)
- Matplotlib, NumPy

On Debian/Ubuntu:
```bash
sudo apt update
sudo apt install soapysdr-tools libsoapysdr-dev python3-soapysdr python3-matplotlib python3-numpy
```

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

SIMULATE = False  # will auto-switch to True if SoapySDR/device not available

# --- Try to import SoapySDR ---
try:
    import SoapySDR
    from SoapySDR import *  # noqa
    have_soapysdr = True
except Exception as e:
    print("SoapySDR import failed:", e)
    have_soapysdr = False
    SIMULATE = True

print("SIMULATE:", SIMULATE)

## Parameters
- Adjust `center_freq`, `sample_rate`, and `gain` to your scenario.
- Set `MAX_UPDATES` to limit the number of screen updates (set to `None` for continuous run).

In [ ]:
# SDR parameters
center_freq = 100.0e6   # 100 MHz
sample_rate = 2.0e6     # 2 MS/s
gain = 30               # in dB

# FFT parameters
fft_size = 2048
window = np.hanning(fft_size)

# Loop control
MAX_UPDATES = 200  # set None to run indefinitely
SLEEP_BETWEEN = 0.02  # seconds pause between updates

# Optional: limit number of channels to use (None = all available)
LIMIT_CHANNELS = None

# For simulation mode
sim_tone_offsets_hz = [ -200e3, -50e3, 0, 75e3, 180e3 ]  # relative to center
sim_snr_db = 30.0
np.set_printoptions(suppress=True)

## Device Setup
Tries to open `driver=krakensdr` first. If not found, will attempt a generic device. Fallback to simulation if none available.

In [ ]:
sdr = None
rx_stream = None
channels = []

def setup_device():
    global sdr, rx_stream, channels, SIMULATE
    if not have_soapysdr:
        SIMULATE = True
        print("Running in SIMULATE mode: SoapySDR not available.")
        return
    
    # Try KrakenSDR first
    try:
        sdr = SoapySDR.Device(dict(driver="krakensdr"))
        print("Opened device: KrakenSDR")
    except Exception as e:
        print("KrakenSDR open failed:", e)
        # Try any available device as a fallback
        try:
            sdr = SoapySDR.Device({})
            print("Opened device: Generic SoapySDR device")
        except Exception as e2:
            print("Generic device open failed:", e2)
            SIMULATE = True
            print("Running in SIMULATE mode.")
            return

    try:
        # Determine channels
        n_channels = sdr.getNumChannels(SOAPY_SDR_RX)
        chan_list = list(range(n_channels))
        if LIMIT_CHANNELS is not None:
            chan_list = chan_list[:LIMIT_CHANNELS]
        channels = chan_list
        print(f"RX channels: {channels}")

        # Configure each channel
        for ch in channels:
            sdr.setSampleRate(SOAPY_SDR_RX, ch, sample_rate)
            sdr.setFrequency(SOAPY_SDR_RX, ch, center_freq)
            try:
                sdr.setGain(SOAPY_SDR_RX, ch, gain)
            except Exception:
                pass

        # Setup stream
        rx_stream = sdr.setupStream(SOAPY_SDR_RX, SOAPY_SDR_CF32, channels)
        sdr.activateStream(rx_stream)
        print("Stream activated.")
    except Exception as e:
        print("Device configuration failed:", e)
        SIMULATE = True
        print("Running in SIMULATE mode.")

setup_device()

## Helpers: FFT & Power
We compute complex FFT with Hann window and convert to dBFS-like power (relative).

In [ ]:
def compute_power_db(x):
    X = np.fft.fftshift(np.fft.fft(x * window))
    p = 20.0 * np.log10(np.abs(X) + 1e-12)
    return p

def simulate_block(num_ch, nsamps, fs, t0=0.0):
    """Generate multi-tone complex samples per channel for testing."""
    t = t0 + np.arange(nsamps) / fs
    sigs = []
    for ch in range(num_ch):
        # Build a few tones with slight channel-dependent offsets
        x = np.zeros(nsamps, dtype=np.complex64)
        for i, foff in enumerate(sim_tone_offsets_hz):
            # slight per-channel detune
            f = foff + (ch * 1e3) + (i * 500)
            x += np.exp(1j * 2 * np.pi * f * t).astype(np.complex64)
        # Add noise based on sim_snr_db (rough)
        sig_power = np.mean(np.abs(x)**2)
        snr_lin = 10**(sim_snr_db/10.0)
        noise_power = sig_power / snr_lin
        noise = (np.sqrt(noise_power/2) * (np.random.randn(nsamps) + 1j*np.random.randn(nsamps))).astype(np.complex64)
        x = x + noise
        # Normalize to prevent clipping of plots
        x /= np.max(np.abs(x) + 1e-9)
        sigs.append(x.astype(np.complex64))
    return sigs


## Live Power Spectra (one figure per channel)
Run the cell below. Stop with **Kernel ▶ Interrupt** when running indefinitely.

In [ ]:
figs = []
axes = []

def setup_figures(num_ch):
    global figs, axes
    figs, axes = [], []
    for ch in range(num_ch):
        fig = plt.figure(figsize=(7, 3))
        ax = fig.add_subplot(111)
        ax.set_title(f"Channel {ch} Power Spectrum")
        ax.set_xlabel("FFT bin")
        ax.set_ylabel("Power (dB)")
        ax.set_ylim(-100, 5)
        figs.append(fig)
        axes.append(ax)

def read_block():
    """Read a block of samples for all channels. Returns list of np.complex64 arrays."""
    if SIMULATE:
        # Generate simulated signals
        # Use either 5 or a typical Kraken channel count when device unknown
        num_ch = len(channels) if channels else 5
        return simulate_block(num_ch, fft_size, sample_rate, t0=time.time())
    else:
        buffs = [np.empty(fft_size, np.complex64) for _ in channels]
        try:
            # Handle both return API styles
            r = sdr.readStream(rx_stream, buffs, fft_size)
            if isinstance(r, tuple):
                ret = r[0]
            else:
                ret = getattr(r, 'ret', None)
            if ret != fft_size:
                # Could be timeout or overflow; return empties to keep loop alive
                return [np.zeros(fft_size, np.complex64) for _ in channels]
            return buffs
        except Exception as e:
            print("readStream error:", e)
            return [np.zeros(fft_size, np.complex64) for _ in channels]

def run_live():
    num_ch = len(channels) if not SIMULATE else (len(channels) if channels else 5)
    setup_figures(num_ch)
    update_count = 0
    try:
        while True:
            sigs = read_block()
            for ch in range(len(sigs)):
                power = compute_power_db(sigs[ch])
                ax = axes[ch]
                ax.cla()
                ax.set_title(f"Channel {ch} Power Spectrum")
                ax.set_xlabel("FFT bin")
                ax.set_ylabel("Power (dB)")
                ax.set_ylim(-100, 5)
                ax.plot(power)
                figs[ch].canvas.draw()
                figs[ch].canvas.flush_events()
            update_count += 1
            if MAX_UPDATES is not None and update_count >= MAX_UPDATES:
                break
            time.sleep(SLEEP_BETWEEN)
    except KeyboardInterrupt:
        print("Stopped by user.")
    finally:
        if not SIMULATE and rx_stream is not None:
            try:
                sdr.deactivateStream(rx_stream)
                sdr.closeStream(rx_stream)
            except Exception:
                pass
        print("Done.")

run_live()

## Optional: Save last spectra to files
Run this cell after stopping the live loop to save the most recent computed spectra to `.npy` files.

In [ ]:
# Example to save one snapshot of spectra
sigs = read_block()
for ch, x in enumerate(sigs):
    power = compute_power_db(x)
    np.save(f"channel_{ch}_power.npy", power)
print("Saved channel_*_power.npy files in current directory.")

## Cleanup
Make sure the stream is closed if you interrupted execution in the middle.

In [ ]:
try:
    if not SIMULATE and rx_stream is not None:
        sdr.deactivateStream(rx_stream)
        sdr.closeStream(rx_stream)
        print("Stream closed.")
except Exception:
    pass